# AgriConnect — MobileNetV2 Leaf Disease Classification Training Pipeline

This notebook trains a transfer-learning convolutional neural network using **MobileNetV2** on the **PlantVillage dataset** (~54,300 images across 38 crop disease classes).

### Workflow Steps:
1. **Environment Setup & GPU Check**
2. **Dataset Download & Stratified Splits (70% Train, 15% Val, 15% Test)**
3. **Data Augmentation Pipeline (Flip, Rotation, Zoom, Contrast)**
4. **MobileNetV2 Architecture Setup with Custom Classification Head**
5. **Phase 1 Training: Frozen Base (5 Epochs)**
6. **Phase 2 Fine-Tuning: Unfreeze Top 30 Layers with Low Learning Rate**
7. **Evaluation (Accuracy, Precision, Recall, Confusion Matrix)**
8. **Export Model & Class Names to `../app/ml/artifacts/v1/`**

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

print(f"TensorFlow Version: {tf.__version__}")
print("GPUs Available:", tf.config.list_physical_devices('GPU'))

## 1. Dataset Configuration
Set path to downloaded PlantVillage dataset directory containing class subfolders.

In [ ]:
DATA_DIR = './data/plantvillage'
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Load full dataset directory
dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    labels='inferred',
    label_mode='categorical',
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

class_names = dataset.class_names
num_classes = len(class_names)
print(f"Found {num_classes} classes:", class_names[:5])

## 2. Train / Val / Test Stratified Split (70% / 15% / 15%)

In [ ]:
dataset_size = tf.data.experimental.cardinality(dataset).numpy()
train_size = int(0.70 * dataset_size)
val_size = int(0.15 * dataset_size)
test_size = dataset_size - train_size - val_size

train_ds = dataset.take(train_size)
remaining_ds = dataset.skip(train_size)
val_ds = remaining_ds.take(val_size)
test_ds = remaining_ds.skip(val_size)

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

## 3. Data Augmentation & Model Construction

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomContrast(0.1),
], name="data_augmentation")

# MobileNetV2 base pretrained on ImageNet
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False  # Freeze for initial training

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
# MobileNetV2 preprocesses inputs from [0, 255] to [-1, 1]
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax', name="predictions")(x)

model = keras.Model(inputs, outputs, name="agriconnect_disease_model")
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)
model.summary()

## 4. Phase 1 Training (Frozen Base, 5 Epochs)

In [ ]:
epochs_phase1 = 5
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_phase1
)

## 5. Phase 2 Fine-Tuning (Unfreeze Top 30 Layers)

In [ ]:
base_model.trainable = True
fine_tune_at = len(base_model.layers) - 30
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top_3_accuracy')]
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
]

epochs_fine = 10
history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs_phase1 + epochs_fine,
    initial_epoch=history_phase1.epoch[-1],
    callbacks=callbacks
)

## 6. Evaluation & Confusion Matrix

In [ ]:
eval_results = model.evaluate(test_ds)
print(f"Test Loss: {eval_results[0]:.4f}")
print(f"Test Accuracy: {eval_results[1]:.4f}")
print(f"Test Top-3 Accuracy: {eval_results[2]:.4f}")

## 7. Model Export to Backend Artifacts

In [ ]:
export_dir = '../app/ml/artifacts/v1'
os.makedirs(export_dir, exist_ok=True)

model_path = os.path.join(export_dir, 'disease_model.keras')
model.save(model_path)
print(f"Successfully exported model to: {model_path}")

# Export class names
class_file = os.path.join(export_dir, 'class_names.json')
with open(class_file, 'w', encoding='utf-8') as f:
    json.dump(class_names, f, indent=2)
print(f"Exported class list to: {class_file}")